In [1]:
import ROOT as rt
import ctypes
%jsroot on
file = rt.TFile("e21062_44S.root")
dtime = file.Get("dtime")

/opt/anaconda3/envs/Sulfur/lib/python3.11/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /opt/anaconda3/envs/Sulfur/include/site/python3.11); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


In [23]:
#44SGated
def decay_Daughter(x, par):
    """
    a = amplitude
    b = decay constant (lambda)
    bkgrd = constant background
    """
    a = par[0]
    b = par[1]
    bkgrd = par[2]
    t = x[0]

    return a * rt.TMath.Exp(-b * t) + bkgrd

def Decayp(hist):
    
    MaxX = hist.GetXaxis().GetXmax();
    MinX = hist.GetXaxis().GetXmin();
    # Axis range
    MinX = 1
    #MaxX = 100
    
    print("Min X: ", MinX)
    print("Max X: ", MaxX)
    
    # Canvas
    fitgraph = rt.TCanvas("fits", "fits", 800, 600)
    fitgraph.Clear()

    # Parent half-life guess
    Parent_halflife = rt.TMath.Log(2) / (3.13e3 + 1)

    # Define TF1
    decayp = rt.TF1("decayp", decay_parent, MinX, MaxX, 3)

    decayp.SetParName(0, "source #")
    decayp.SetParName(1, "Source lambda")
    decayp.SetParName(2, "shift")

    decayp.SetParameters(600, Parent_halflife, 400) # intial conditions

    decayp.SetParLimits(0, 0, 1000) # source
    decayp.SetParLimits(1, 0, 1) # lambda
    decayp.SetParLimits(2, 0, 800) # bckgrnd

    # Print initial parameters
    for i in range(decayp.GetNpar()):
        print(f"Initial p{i} = {decayp.GetParameter(i)}")

    # Perform fit
    hist.Fit("decayp", "RS", "", MinX, MaxX)
    hist.Print("V")
    print("\n")
    
    # Print parameter limits
    for i in range(decayp.GetNpar()):
        par_min = ctypes.c_double()
        par_max = ctypes.c_double()
        decayp.GetParLimits(i, par_min, par_max)
        print(f"Parameter {i} ({decayp.GetParName(i)}) limits: [{par_min}, {par_max}]")

    # Half-life calculation
    lambda_fit = decayp.GetParameter(1)
    lambda_err = decayp.GetParError(1)

    decayhl = rt.TMath.Log(2) / lambda_fit
    decay_error = (rt.TMath.Log(2) / (lambda_fit**2)) * lambda_err

    print(f"Decay: {decayhl:.6f} ms ± {decay_error:.6f} ms")

    # Chi2 / NDF
    chi2 = decayp.GetChisquare()
    ndf = decayp.GetNDF()
    print(f"Chi^2/NDF: {chi2/ndf:.6f}\n")

    # Constant background function
    bckgrnd = rt.TF1("bckgrnd", "[0]", MinX, MaxX)
    bckgrnd.SetParameter(0, decayp.GetParameter(2))
    bckgrnd.SetLineColor(rt.kBlack)

    # Legend
    legend = rt.TLegend(0.7, 0.4, 0.9, 0.7)
    legend.AddEntry(hist, "Data", "E")
    legend.AddEntry(decayp, "Fit", "l")
    legend.AddEntry(bckgrnd, "backgrnd", "l")

    # Styling
    hist.SetTitle("Decay of 44Ar Gated on 1157 keV;Time (ms);Counts")
    hist.GetXaxis().CenterTitle(True)
    hist.GetYaxis().CenterTitle(True)
    hist.SetLineColor(rt.kBlue)

    # Draw
    hist.Draw("E")
    decayp.SetLineColor(rt.kRed)
    decayp.Draw("same")
    bckgrnd.Draw("same")
    fitgraph.Draw()
    hist.Print("V")
    # legend.Draw()  # Uncomment if desired

    fitgraph.SaveAs("Gamma_1157.png")

In [27]:
gamma_canvas = rt.TCanvas("gamma_canvas", "gamma_1158")
gamma_1158 = dtime.ProjectionX("Decay_Curve_1157keV",1154,1161)
Decayp(gamma_1158)
gamma_1158.Draw()
gamma_canvas.Update()
gamma_canvas.Draw()

Min X:  1
Max X:  1000.0
Initial p0 = 600.0
Initial p1 = 0.0002213820442542144
Initial p2 = 400.0
Parameter 0 (source #) limits: [c_double(0.0), c_double(1000.0)]
Parameter 1 (Source lambda) limits: [c_double(0.0), c_double(1.0)]
Parameter 2 (shift) limits: [c_double(0.0), c_double(800.0)]
Decay: 9160.438914 ms ± 0.000000 ms

Chi^2/NDF: 1.037451

****************************************
         Invalid FitResult  (status = 4 )
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =       1033.3
NDf                       =          996
Edm                       =  7.18184e-07
NCalls                    =         1352
source #                  =      856.056   +/-   0            	 (limited)
Source lambda             =  7.56675e-05   +/-   0            	 (limited)
shift                     =  6.80857e-06   +/-   0            	 (limited)
TH1.Print Name  = Decay_Curve_1157keV, Entries= 826257, Total sum= 825479
TH1.Print Name  = Decay_Curve_1157keV

Warning in <TCanvas::Constructor>: Deleting canvas with same name: gamma_canvas
Warning in <Fit>: Abnormal termination of minimization.
Info in <TCanvas::Print>: png file Gamma_1157.png has been created


In [21]:
bckgrnd_canvas = rt.TCanvas("bckgrnd_canvas", "gamma_2789")
bckgrnd_2789 = dtime.ProjectionX("Decay_Curve_2789keV",2810,2819)
bckgrnd_2789.Draw()
bckgrnd_canvas.Draw()

Warning in <TCanvas::Constructor>: Deleting canvas with same name: bckgrnd_canvas


In [22]:
bckgrnd = rt.TF1("bckgrnd","[0]",0,100, 1)
bckgrnd.SetParameters(35)
bckgrnd.SetParLimits(0, 10, 35) # bckgrnd
bckgrnd_2789.Fit("bckgrnd","RS","",1,100)
bckgrnd_2789.Print("V")
bckgrnd_2789.Draw()
bckgrnd_canvas.Draw()

****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      134.107
NDf                       =           98
Edm                       =  2.49262e-11
NCalls                    =           36
p0                        =      22.3727   +/-   0.475266     	 (limited)
TH1.Print Name  = Decay_Curve_2789keV, Entries= 21146, Total sum= 21128
